# Nepal Maize Drought Q1 -- Full Automated Pipeline

**This notebook works in both Google Colab and local Jupyter/VS Code** -- it auto-detects which environment you're in and adjusts file upload and path handling accordingly. No hardcoded usernames or paths.

**Run every cell top to bottom, in order.** Each cell prints a clear PASS/FAIL/warning so you always know what to do next.

What this notebook does:
1. Detects Colab vs local and sets up a safe working folder
2. Installs all required packages
3. Lets you upload/select `Nepal_Maize_Drought_Q1.zip` and verifies it's the latest script versions
4. Walks you through credentials for CDS (ERA5-Land), NASA Earthdata (SMAP), and Google Earth Engine (Sentinel-1/2)
5. Runs the full download pipeline (scripts 01-09), continuing past individual failures and reporting a clear summary
6. Packages the results for you to download or find locally

## 1. Environment detection and safe working folder

In [8]:
import os, sys, platform

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

IS_COLAB = in_colab()

if IS_COLAB:
    BASE_DIR = "/content"
else:
    # Dedicated subfolder under the user's home directory -- NEVER write
    # directly into the home directory root (Windows blocks this with
    # PermissionError, and it's poor practice on any OS).
    BASE_DIR = os.path.join(os.path.expanduser("~"), "NepalMaizeDroughtWork")

os.makedirs(BASE_DIR, exist_ok=True)
PROJECT_ROOT = os.path.join(BASE_DIR, "Nepal_Maize_Drought_Q1")
CODE_DIR = os.path.join(PROJECT_ROOT, "07_code")

print(f"Environment: {'Google Colab' if IS_COLAB else 'Local (' + platform.system() + ')'}")
print(f"Python: {sys.version}")
print(f"Base working directory: {BASE_DIR}")
print(f"Project will be at: {PROJECT_ROOT}")

Environment: Local (Windows)
Python: 3.10.19 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 16:41:31) [MSC v.1929 64 bit (AMD64)]
Base working directory: C:\Users\user\NepalMaizeDroughtWork
Project will be at: C:\Users\user\NepalMaizeDroughtWork\Nepal_Maize_Drought_Q1


## 2. Install dependencies

In [12]:
!"{sys.executable}" -m pip install -q cdsapi earthaccess earthengine-api geopandas shapely rasterio xarray netCDF4 rioxarray climate-indices scikit-learn xgboost lightgbm shap statsmodels matplotlib pandas requests
print("Dependencies installed.")

Dependencies installed.


## 3. Upload/locate the project zip and extract it
This works whether you're in Colab (browser upload) or local Jupyter/VS Code (native file picker).

In [44]:
import os

# Point directly at your already-unzipped folder
PROJECT_ROOT = r"C:\Users\user\Downloads\Nepal_Maize_Drought_Q1\Nepal_Maize_Drought_Q1"
CODE_DIR = os.path.join(PROJECT_ROOT, "07_code")

if not os.path.isdir(PROJECT_ROOT):
    raise RuntimeError(f"Folder not found: {PROJECT_ROOT}. Double-check the path/spelling.")
if not os.path.isdir(CODE_DIR):
    raise RuntimeError(f"Found {PROJECT_ROOT} but no 07_code subfolder inside it -- "
                        f"check you unzipped it correctly (contents: {os.listdir(PROJECT_ROOT)})")

os.environ["NEPAL_DROUGHT_CODE_DIR"] = CODE_DIR
os.chdir(CODE_DIR)
print("Now in:", os.getcwd())
for entry in sorted(os.listdir(CODE_DIR)):
    print(" ", entry)

# Verify you have the latest (v3, monthly-chunked) ERA5-Land script
era5_script = os.path.join(CODE_DIR, "02_download_era5_land.py")
with open(era5_script) as f:
    era5_content = f.read()

if "MONTHLY_DIR" in era5_content:
    print("\n✅ Confirmed: latest v3 (monthly-chunked) ERA5-Land script is in place.")
else:
    raise RuntimeError(
        "⚠️  This is an OLDER version of 02_download_era5_land.py (no monthly chunking) -- "
        "it WILL hit the 'cost limits exceeded' error. Re-download the latest "
        "Nepal_Maize_Drought_Q1.zip from the chat, delete this folder, and re-unzip."
    )

Now in: C:\Users\user\Downloads\Nepal_Maize_Drought_Q1\Nepal_Maize_Drought_Q1\07_code
  00_setup_check.py
  01_get_boundaries.py
  02_download_era5_land.py
  03_download_chirps.py
  04_download_smap.py
  05_download_soilgrids.py
  06_download_sentinel_gee.py
  07_compute_spi_spei.py
  08_build_temporal_coverage_matrix.py
  09_data_quality_checks.py
  __pycache__
  fix_02_download_era5_land_climatology.py
  project_config.py
  yield_moald_recovery_notes.md

✅ Confirmed: latest v3 (monthly-chunked) ERA5-Land script is in place.


In [10]:
# Verify the ERA5-Land script is the latest (v3, monthly-chunked) version --
# this is the single most common source of confusing errors in this pipeline.
era5_script = os.path.join(CODE_DIR, "02_download_era5_land.py")
with open(era5_script) as f:
    era5_content = f.read()

if "MONTHLY_DIR" in era5_content:
    print("\u2705 Confirmed: latest v3 (monthly-chunked) ERA5-Land script is in place.")
else:
    raise RuntimeError(
        "\u26a0\ufe0f  This is an OLDER version of 02_download_era5_land.py (no monthly "
        "chunking) -- it WILL hit the 'cost limits exceeded' error. Re-download the "
        "latest Nepal_Maize_Drought_Q1.zip from the chat and re-run the upload cell above."
    )

✅ Confirmed: latest v3 (monthly-chunked) ERA5-Land script is in place.


## 4. Setup check (what's configured, what isn't -- nothing downloads yet)

In [16]:
!"{sys.executable}" 00_setup_check.py

SETUP CHECK
[OK ] Copernicus CDS API credentials (~/.cdsapirc)
[MISSING] NASA Earthdata login (~/.netrc entry for urs.earthdata.nasa.gov)
          -> Register at https://urs.earthdata.nasa.gov, then run `python -c "import earthaccess; earthaccess.login(persist=True)"` to write ~/.netrc automatically.
[OK ] Google Earth Engine authentication
[MISSING] EE_PROJECT environment variable (Cloud project for ee.Initialize)
          -> Create a Cloud project at https://console.cloud.google.com/projectcreate, register it for Earth Engine at https://code.earthengine.google.com/register, then set `os.environ['EE_PROJECT'] = 'your-project-id'` (notebook) or `export EE_PROJECT=your-project-id` (shell) before running 06_download_sentinel_gee.py.
[OK ] Python package: cdsapi
[OK ] Python package: earthaccess
[OK ] Python package: ee
[OK ] Python package: geopandas
[OK ] Python package: xarray
[OK ] Python package: climate_indices
[OK ] Python package: sklearn
Some credentials/packages are missing (s

c:\Users\user\miniconda3\envs\xo_pipeline\lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


## 5. Credentials (run each cell once per session)

- **CDS (ERA5-Land)**: get/regenerate your key at https://cds.climate.copernicus.eu/profile.
  **Also required once**: accept the license at
  https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land (scroll down, click Accept) --
  every download fails without this, regardless of key validity.
- **NASA Earthdata (SMAP)**: register free at https://urs.earthdata.nasa.gov
- **Google Earth Engine (Sentinel-1/2)**: create a Cloud project at
  https://console.cloud.google.com/projectcreate, then register it (free, non-commercial)
  at https://code.earthengine.google.com/register

In [18]:
# 5a. Copernicus CDS API key
import getpass

cdsapirc_path = os.path.join(os.path.expanduser("~"), ".cdsapirc")
if os.path.exists(cdsapirc_path):
    print(f"{cdsapirc_path} already exists. Delete it first if you need to enter a new key.")
else:
    cds_key = getpass.getpass("Paste your CDS API key (input hidden): ").strip()
    with open(cdsapirc_path, "w") as f:
        f.write(f"url: https://cds.climate.copernicus.eu/api\nkey: {cds_key}\n")
    del cds_key
    print(f"{cdsapirc_path} written.")

print("\nReminder: have you accepted the license at "
      "https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land yet? "
      "If not, do that now in your browser before running the pipeline.")

C:\Users\user\.cdsapirc already exists. Delete it first if you need to enter a new key.

Reminder: have you accepted the license at https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land yet? If not, do that now in your browser before running the pipeline.


In [22]:
# 5b. NASA Earthdata login (for SMAP)
import earthaccess
auth = earthaccess.login(persist=True)  # prompts for username/password interactively
print("Authenticated:", auth.authenticated)
if not auth.authenticated:
    print("Login failed -- re-run this cell and double check your username/password at "
          "https://urs.earthdata.nasa.gov")

Authenticated: True


In [20]:
# 5c. Google Earth Engine
import ee

EE_PROJECT = input("Paste your Earth Engine Cloud project ID: ").strip()
os.environ["EE_PROJECT"] = EE_PROJECT

try:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
    print(f"\u2705 Earth Engine initialized with project: {EE_PROJECT}")
except Exception as e:
    print(f"Earth Engine init failed: {e}")
    print("Check: (1) the project ID is correct, (2) you registered it at "
          "https://code.earthengine.google.com/register")

c:\Users\user\miniconda3\envs\xo_pipeline\lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)



Successfully saved authorization token.


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


✅ Earth Engine initialized with project: prime-agency-509009-q5


SyntaxError: invalid syntax (3318071852.py, line 1)

In [21]:
# Re-check now that credentials are in place
!"{sys.executable}" 00_setup_check.py

SETUP CHECK
[OK ] Copernicus CDS API credentials (~/.cdsapirc)
[MISSING] NASA Earthdata login (~/.netrc entry for urs.earthdata.nasa.gov)
          -> Register at https://urs.earthdata.nasa.gov, then run `python -c "import earthaccess; earthaccess.login(persist=True)"` to write ~/.netrc automatically.
[OK ] Google Earth Engine authentication
[OK ] EE_PROJECT environment variable (Cloud project for ee.Initialize)
[OK ] Python package: cdsapi
[OK ] Python package: earthaccess
[OK ] Python package: ee
[OK ] Python package: geopandas
[OK ] Python package: xarray
[OK ] Python package: climate_indices
[OK ] Python package: sklearn
Some credentials/packages are missing (see above). Scripts that need them will refuse to run with fabricated data -- fix the items above first, or skip that specific source and note the gap in 08_metadata/processing_log.csv.


c:\Users\user\miniconda3\envs\xo_pipeline\lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


## 6. Run the full pipeline (scripts 01 through 09, in order)

This captures and prints EVERY script's full output (stdout + stderr), so errors are never
silently swallowed. It continues past individual script failures and gives you a clear
PASS/FAIL summary at the end. This step can take a long time (ERA5-Land and Sentinel
downloads are large) -- let it run.

In [1]:
import os, subprocess, sys

# --- Re-establish paths (needed again after any kernel restart) ---
PROJECT_ROOT = r"C:\Users\user\Downloads\Nepal_Maize_Drought_Q1\Nepal_Maize_Drought_Q1"
CODE_DIR = os.path.join(PROJECT_ROOT, "07_code")
BASE_DIR = r"C:\Users\user\Downloads\Nepal_Maize_Drought_Q1"

if not os.path.isdir(CODE_DIR):
    raise RuntimeError(f"07_code not found at {CODE_DIR} -- check the path is correct.")

os.environ["NEPAL_DROUGHT_CODE_DIR"] = CODE_DIR
os.chdir(CODE_DIR)
print("Working directory:", os.getcwd())

# --- Run the setup check first, so missing credentials are obvious up front ---
print("\n" + "=" * 72)
print("SETUP CHECK")
print("=" * 72)
proc = subprocess.Popen([sys.executable, "-u", "00_setup_check.py"],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    print("\n⚠️  Some credentials are missing (see [MISSING] lines above). Go back and "
          "run the matching credential cell(s) -- CDS key / NASA Earthdata login / "
          "Earth Engine project -- before continuing, or the matching download "
          "script below will fail.\n")

# --- Run the full pipeline ---
SCRIPTS = [
    "01_get_boundaries.py",
    "02_download_era5_land.py",
    "03_download_chirps.py",       # large download -- comment out to skip; optional cross-check only
    "04_download_smap.py",
    "05_download_soilgrids.py",
    "06_download_sentinel_gee.py",
    "07_compute_spi_spei.py",       # EXPECTED to be BLOCKED on a fresh run -- see note below
    "08_build_temporal_coverage_matrix.py",
    "09_data_quality_checks.py",
]

results = []
for script in SCRIPTS:
    print("=" * 72)
    print(f"RUNNING {script}")
    print("=" * 72)

    proc = subprocess.Popen(
        [sys.executable, "-u", script],   # -u = unbuffered, so prints show up immediately
        env=os.environ.copy(),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,         # merge stderr into stdout, in the order it's printed
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()

    status = "PASS" if proc.returncode == 0 else f"FAILED (exit {proc.returncode})"
    results.append((script, status))
    print(f"\n--- {script}: {status} ---\n")

print("\n" + "=" * 72)
print("PIPELINE SUMMARY")
print("=" * 72)
for script, status in results:
    print(f"  [{status:22s}] {script}")
print("\nNOTE: 07_compute_spi_spei.py is EXPECTED to fail/BLOCK on a fresh run -- it needs a\n"
      "separate long-record (1991-2024) precipitation climatology that\n"
      "02_download_era5_land.py does not pull by default (see that script's docstring).\n"
      "This is intentional: it refuses to compute SPI/SPEI against too-short a record\n"
      "rather than producing misleading values.\n"
      "\nIf any OTHER script shows FAILED, scroll up to see exactly where in that "
      "script's live output it failed.")

Working directory: C:\Users\user\Downloads\Nepal_Maize_Drought_Q1\Nepal_Maize_Drought_Q1\07_code

SETUP CHECK
c:\Users\user\miniconda3\envs\xo_pipeline\lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
SETUP CHECK
[OK ] Copernicus CDS API credentials (~/.cdsapirc)
[MISSING] NASA Earthdata login (~/.netrc entry for urs.earthdata.nasa.gov)
          -> Register at https://urs.earthdata.nasa.gov, then run `python -c "import earthaccess; earthaccess.login(persist=True)"` to write ~/.netrc automatically.
[OK ] Google Earth Engine authentication
[MISSING] EE_PROJECT environment variable (Cloud project for ee.Initialize)
          -> Cr

## 7. Fill in the manual yield data (the one step that cannot be automated)

No public API exists for official district-year MoALD/CBS maize yield records. Open
`01_raw_data/yield_moald/yield_moald_TEMPLATE.csv` (via the file browser in Colab's left
sidebar, or directly on disk locally) and follow
`07_code/yield_moald_recovery_notes.md` for where to look. This can be done later --
it doesn't block the rest of the pipeline.

In [34]:
print(f"Yield template location: {os.path.join(PROJECT_ROOT, '01_raw_data', 'yield_moald', 'yield_moald_TEMPLATE.csv')}")
print(f"Instructions:            {os.path.join(CODE_DIR, 'yield_moald_recovery_notes.md')}")

Yield template location: C:\Users\user\Downloads\Nepal_Maize_Drought_Q1\Nepal_Maize_Drought_Q1\01_raw_data\yield_moald\yield_moald_TEMPLATE.csv
Instructions:            C:\Users\user\Downloads\Nepal_Maize_Drought_Q1\Nepal_Maize_Drought_Q1\07_code\yield_moald_recovery_notes.md


## 8. Package the results

In [ ]:
results_zip_base = os.path.join(BASE_DIR, "Nepal_Maize_Drought_Q1_RESULTS")
if os.path.exists(results_zip_base + ".zip"):
    os.remove(results_zip_base + ".zip")
shutil.make_archive(results_zip_base, "zip", PROJECT_ROOT)
results_zip = results_zip_base + ".zip"
print(f"Saved: {results_zip}")

if IS_COLAB:
    from google.colab import files as colab_files
    colab_files.download(results_zip)
else:
    print("This file is already on your computer at the path above -- no download step needed.")
    print("Upload it back in the chat when you're ready to continue with the analysis.")

NameError: name 'shutil' is not defined

In [49]:
python 07_code/fix_02_download_era5_land_climatology.pypython 07_code/fix_02_download_era5_land_climatology.py

SyntaxError: invalid decimal literal (4098972539.py, line 1)